# Notebook 02 — Modelo Preditivo de Risco de Defasagem
### Datathon Fase 5 · Associação Passos Mágicos · Pós-Tech FIAP

**Pergunta 9 (enunciado literal):** *"Quais padrões nos indicadores permitem identificar alunos em risco **antes** de queda no desempenho ou aumento da defasagem? Construa um modelo preditivo que mostre uma **probabilidade** do aluno entrar em **risco de defasagem**."*

**Leitura do enunciado (decisão de arquitetura):**
- O documento nomeia o alvo do modelo, duas vezes (pergunta 9 + seção de entrega), como **"risco de defasagem"**. Logo, o alvo é o **agravamento da defasagem de T para T+1**.
- A pergunta *"quais padrões identificam risco"* **é respondida pelo próprio modelo**, via importância de variáveis / SHAP. Não há análise descritiva desconectada: o modelo É o instrumento analítico.
- *"Queda no desempenho"* entra como **lente de interpretação** (quais indicadores comportamentais/psicossociais pesam no risco), não como segundo alvo.

**Natureza temporal:** features do aluno no ano **T** → prevê agravamento de defasagem em **T+1**. Prospectivo, sem vazamento.


In [ ]:
# ==== SETUP (Colab) ====
# Monta o Drive e autoriza a conta Google.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
import warnings; warnings.filterwarnings('ignore')
sns.set_style('whitegrid'); plt.rcParams['figure.figsize']=(8,5)
RANDOM_STATE = 42

In [ ]:
# ==== CONFIGURAÇÃO (todos os parâmetros de decisão ficam aqui) ====
import os

# Pasta do projeto no Drive (ajustar se o nome for outro).
PASTA_DRIVE = '/content/drive/MyDrive/Datathon'

# Fora do Colab, usa o diretorio atual.
if not EM_COLAB or not os.path.isdir(PASTA_DRIVE):
    PASTA_DRIVE = '.'

CAMINHO_BASE = os.path.join(PASTA_DRIVE, 'painel_modelo.csv')
print('Lendo de:', CAMINHO_BASE, '| existe?', os.path.isfile(CAMINHO_BASE))

# Split: 'por_RA' (GroupShuffleSplit, nenhum aluno em treino+teste)
#        'temporal' (treina 2022->2023, testa 2023->2024)
ESTRATEGIA_SPLIT = 'por_RA'
# 'por_RA' é o regime principal (modelo final e faixas); o corte temporal 2022+23->2024
# é sempre reportado na seção 11, independentemente desta escolha.
# IPP mantido como feature: reconstrução algébrica exata e documentada (sensibilidade na seção 11).
TEST_SIZE = 0.25

# LIMIAR_IDA so afeta a leitura de desempenho (secao 8), nao o alvo.
LIMIAR_IDA = 0.5

# Estratégia de faixas de risco (saída para a ONG):
#   'base_rate' -> Alto >= 2x a base rate, Baixo < base rate
#   'percentil'  -> Alto = top 10%, Medio = 10-30%
ESTRATEGIA_FAIXAS = 'base_rate' 

## 1. Carga da base de modelagem

In [ ]:
df = pd.read_csv(CAMINHO_BASE)
base = df[df['excluir_modelo'] == False].copy()
print('Linhas na base de modelagem:', len(base))
print(base['Ano'].value_counts().sort_index())

## 2. Harmonização de categóricas entre anos
`Gênero` alterna *Menina/Feminino*; `Instituicao` muda de nome por ano. Colapsamos para categorias estáveis.

In [ ]:
base['Genero_pad'] = base['Gênero'].replace(
    {'Menina':'F','Feminino':'F','Menino':'M','Masculino':'M'})
def _rede(x):
    if pd.isna(x): return 'Desconhecida'
    return 'Publica' if 'ública' in str(x) or 'Publica' in str(x) else 'Particular'
base['rede'] = base['Instituicao'].apply(_rede)
print(base['Genero_pad'].value_counts(dropna=False).to_dict())
print(base['rede'].value_counts(dropna=False).to_dict())

## 3. Construção do alvo temporal — agravamento de defasagem (T → T+1)
Para cada aluno presente em anos consecutivos, marcamos `alvo = 1` se a defasagem **piora** (fica mais negativa) do ano T para T+1. As features vêm do ano T; o alvo, da comparação com T+1. Sem vazamento.

In [ ]:
base = base.sort_values(['RA','Ano'])
prox = base[['RA','Ano','Defasagem','IDA']].rename(
    columns={'Defasagem':'Defasagem_T1','IDA':'IDA_T1'})
prox['Ano_join'] = prox['Ano'] - 1
prox = prox.drop(columns='Ano')
pares = base.merge(prox, left_on=['RA','Ano'], right_on=['RA','Ano_join'])

pares['alvo'] = (pares['Defasagem_T1'] < pares['Defasagem']).astype(int)

# Queda de IDA nao entra no alvo; so aparece na leitura da secao 8.
pares['queda_IDA'] = ((pares['IDA'] - pares['IDA_T1']) >= LIMIAR_IDA).astype(int)

print('Transições (pares T->T+1):', len(pares), '| RAs únicos:', pares['RA'].nunique())
print(f"Base rate (agravamento de defasagem): {pares['alvo'].mean():.1%}")
_par = pares['Ano'].astype(str)+'->'+(pares['Ano']+1).astype(str)
print('\nPor par de anos:')
print(pares.groupby(_par)['alvo'].agg(['count','mean']).round(3))

### 3.1 Por que a defasagem tem forte componente estrutural
O agravamento de defasagem depende fortemente da defasagem **atual**: quem está no nível certo (defasagem 0) tem espaço para cair; quem já está muito defasado quase não piora (piso). Isso é **sinal real** (não vazamento — a defasagem de T é passado em relação a T+1), mas precisa ser reconhecido: parte da performance do modelo vem dessa mecânica estrutural.

In [ ]:
tab = pares.groupby('Defasagem')['alvo'].agg(['mean','count']).round(3)
tab.columns = ['taxa_agravamento','n']
print(tab)

## 4. Features
Conjunto **parcimonioso e sem redundância** (decisão validada empiricamente):
- **Estruturais:** `Defasagem`, `fase_num` — a mecânica da progressão.
- **Acadêmico/comportamentais/psicossociais:** `IDA, IEG, IAA, IPS, IPV, IPP` — os 6 índices puros.
- **Categóricas:** `Genero_pad`, `rede`.

**Excluídas de propósito (com motivo):**
- `IAN` — função-degrau da Defasagem → vazamento.
- `INDE_ano` — é média ponderada dos próprios índices (r≈0,75 com IDA); redundante e contamina o SHAP. Fica para validação/descritivo.
- `idade_padronizada` — r≈0,90 com `fase_num`; mantemos só `fase_num` (mais interpretável para a ONG).
- Features de tendência (variação ano-a-ano) — testadas, não agregam AUC (+0,001). Removidas por parcimônia.

In [ ]:
num_features = ['Defasagem','fase_num','IDA','IEG','IAA','IPS','IPV','IPP']
cat_features = ['Genero_pad','rede']

X = pares[num_features + cat_features].copy()
y = pares['alvo'].copy()
groups = pares['RA'].copy()
print('Shape X:', X.shape, '| NaN:', int(X.isna().sum().sum()),
      '| base rate:', round(y.mean(),3))

## 5. Separação treino/teste (sem vazamento de grupo)

In [ ]:
if ESTRATEGIA_SPLIT == 'por_RA':
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    itr, ite = next(gss.split(X, y, groups))
    Xtr, Xte, ytr, yte = X.iloc[itr], X.iloc[ite], y.iloc[itr], y.iloc[ite]
    gtr = groups.iloc[itr]
    leak = set(groups.iloc[itr]) & set(groups.iloc[ite])
    print('Split POR_RA | RAs sobrepostos:', len(leak), '(deve ser 0)')
elif ESTRATEGIA_SPLIT == 'temporal':
    mtr, mte = pares['Ano']==2022, pares['Ano']==2023
    Xtr, Xte, ytr, yte = X[mtr], X[mte], y[mtr], y[mte]
    gtr = groups[mtr]
    print('Split TEMPORAL | treino=2022->2023, teste=2023->2024')
print(f'Treino: {len(Xtr)} | Teste: {len(Xte)} | base rate treino {ytr.mean():.3f} / teste {yte.mean():.3f}')

## 6. Treino e comparação de modelos

In [ ]:
preproc = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)])

modelos = {
 'Regressão Logística': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
 'Random Forest': RandomForestClassifier(n_estimators=400, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE),
 'HistGradientBoosting': HistGradientBoostingClassifier(random_state=RANDOM_STATE),
}
resultados = {}
for nome, clf in modelos.items():
    pipe = Pipeline([('preproc', preproc), ('clf', clf)])
    pipe.fit(Xtr, ytr)
    proba = pipe.predict_proba(Xte)[:, 1]
    resultados[nome] = {'pipe': pipe, 'proba': proba,
        'roc_auc': roc_auc_score(yte, proba),
        'pr_auc': average_precision_score(yte, proba)}
tab = pd.DataFrame({k:{'ROC-AUC':v['roc_auc'],'PR-AUC':v['pr_auc']} for k,v in resultados.items()}).T.round(3)
print(tab.sort_values('ROC-AUC', ascending=False))

In [ ]:
# Robustez: validação cruzada agrupada por RA no melhor modelo
melhor_nome = tab['ROC-AUC'].idxmax()
melhor_pipe = Pipeline([('preproc', preproc), ('clf', modelos[melhor_nome])])
scores = cross_val_score(melhor_pipe, X, y, groups=groups, cv=GroupKFold(5), scoring='roc_auc')
print(f'Melhor modelo: {melhor_nome}')
print(f'ROC-AUC 5-fold (agrupado por RA): {scores.mean():.3f} +/- {scores.std():.3f}')
print('Folds:', np.round(scores,3))

## 7. Avaliação do modelo escolhido

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
melhor = resultados[melhor_nome]
proba = melhor['proba']

# Limiar para a classificacao binaria reportada. 0.30 maximiza a balanced accuracy
# no metodo out-of-fold (bal_acc 0.78, acc 0.84). O corte 'Alto' das faixas (secao 10)
# usa 2x a base rate e serve a outro proposito (priorizacao gerencial).
LIMIAR_DECISAO = 0.30
pred = (proba >= LIMIAR_DECISAO).astype(int)

acc = accuracy_score(yte, pred)
bacc = balanced_accuracy_score(yte, pred)
baseline = 1 - yte.mean()
print(f'=== {melhor_nome} (limiar {LIMIAR_DECISAO}) ===\n')
print(f'Acuracia:          {acc:.3f}   (minimo exigido: 0.75)')
print(f'Balanced accuracy: {bacc:.3f}')
print(f'Baseline trivial:  {baseline:.3f}   (prever sempre "nao agrava")')
print(f'F1 (agrava):       {f1_score(yte, pred):.3f}')
print(f'ROC-AUC:           {melhor["roc_auc"]:.3f}\n')
print(classification_report(yte, pred, digits=3, zero_division=0,
      target_names=['Nao agrava (0)','Agrava defasagem (1)']))
# A acuracia sozinha engana num alvo desbalanceado (baseline ~0.80). A balanced accuracy
# confirma que o modelo separa as duas classes acima do acaso.

fig, ax = plt.subplots(1,2,figsize=(13,5))
fpr,tpr,_ = roc_curve(yte, proba)
ax[0].plot(fpr,tpr,label=f"AUC={melhor['roc_auc']:.3f}"); ax[0].plot([0,1],[0,1],'--',color='gray')
ax[0].set_title('Curva ROC'); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].legend()
ConfusionMatrixDisplay(confusion_matrix(yte,pred),
    display_labels=['Não agrava','Agrava']).plot(ax=ax[1], colorbar=False)
ax[1].set_title(f'Matriz de confusão (limiar {LIMIAR_DECISAO:.2f})')
plt.tight_layout(); plt.show()

## 8. Decomposição estrutural vs. acionável — *quais padrões identificam risco*
Esta seção responde a primeira frase da pergunta 9. Separamos o poder preditivo em duas camadas:
- **Estrutural** (`Defasagem`, `fase_num`): a mecânica da progressão — previsível por construção.
- **Comportamental/psicossocial** (`IDA, IEG, IAA, IPS, IPV, IPP`): o sinal **acionável** — onde intervenções pedagógicas e psicossociais podem atuar.

Medimos o **incremento de AUC** que os indicadores comportamentais trazem *sobre* a base estrutural. É essa diferença — não o AUC bruto — que mostra o valor do acompanhamento integral da ONG.

In [ ]:
estrutural = ['Defasagem','fase_num']
comport = ['IDA','IEG','IAA','IPS','IPV','IPP']
def auc_grupo(num, cat):
    steps=[('n',StandardScaler(),num)]
    if cat: steps.append(('c',OneHotEncoder(handle_unknown='ignore'),cat))
    pipe=Pipeline([('p',ColumnTransformer(steps)),
                   ('m',RandomForestClassifier(n_estimators=400,class_weight='balanced',
                                               n_jobs=-1,random_state=RANDOM_STATE))])
    s=cross_val_score(pipe, pares[num+cat], y, groups=groups, cv=GroupKFold(5), scoring='roc_auc')
    return s.mean(), s.std()

a0,d0 = auc_grupo(estrutural, [])
a1,d1 = auc_grupo(estrutural+comport, cat_features)
print(f'Só estrutural:               AUC {a0:.3f} ± {d0:.3f}')
print(f'Estrutural + comportamental: AUC {a1:.3f} ± {d1:.3f}')
print(f'\nINCREMENTO comportamental/psicossocial: {a1-a0:+.3f} de AUC')
print('-> É o ganho preditivo atribuível aos índices em que a ONG pode intervir.')

### 8.1 Lente de desempenho (queda de IDA)
Verificação complementar: os indicadores psicossociais também antecipam **queda de desempenho** (IDA), não só defasagem? Reportamos a taxa de queda de IDA por faixa dos índices psicossociais no ano T.

In [ ]:
for ind in ['IPS','IPP','IEG']:
    q = pd.qcut(pares[ind], 3, labels=['baixo','médio','alto'], duplicates='drop')
    taxa = pares.groupby(q, observed=True)['queda_IDA'].mean().round(3)
    print(f'{ind:4s} (ano T) -> taxa de queda de IDA em T+1 por faixa: {taxa.to_dict()}')

## 9. SHAP — quais indicadores mais pesam na previsão de risco

In [ ]:
import shap
rf_pipe = resultados[melhor_nome]['pipe'] if melhor_nome=='Random Forest' else \
          Pipeline([('preproc',preproc),('clf',modelos['Random Forest'])]).fit(Xtr,ytr)
Xte_proc = rf_pipe.named_steps['preproc'].transform(Xte)
feat_names = num_features + list(
    rf_pipe.named_steps['preproc'].named_transformers_['cat'].get_feature_names_out(cat_features))
clf_rf = rf_pipe.named_steps['clf']
expl = shap.TreeExplainer(clf_rf)
sv = expl.shap_values(Xte_proc)
sv1 = sv[1] if isinstance(sv, list) else (sv[:,:,1] if sv.ndim==3 else sv)
shap.summary_plot(sv1, Xte_proc, feature_names=feat_names, show=False, plot_type='bar')
plt.title('Importância média (|SHAP|) — risco de defasagem'); plt.tight_layout(); plt.show()

# Permutation importance para conferir a leitura do SHAP
r = permutation_importance(rf_pipe, Xte, yte, n_repeats=15, random_state=RANDOM_STATE, scoring='roc_auc')
imp = pd.DataFrame({'feature': num_features+cat_features,
                    'perm_importance': r.importances_mean}).sort_values('perm_importance', ascending=False)
print('Permutation importance (queda de AUC ao embaralhar a feature):')
print(imp.to_string(index=False))

## 10. Faixas de risco (saída gerencial para a ONG)
Os cortes das faixas **não são fixados arbitrariamente** (ex.: 0,33/0,66) — são **derivados dos dados**, para produzir uma ferramenta operacionalmente útil.

Usamos probabilidades **out-of-fold** (cada aluno é pontuado por um modelo que não o viu no treino), o que dá uma estimativa honesta do risco. Duas estratégias de corte, parametrizáveis em `ESTRATEGIA_FAIXAS`:
- **`base_rate` (recomendada):** `Alto` = risco ≥ 2× a média da base; `Baixo` = risco < média. Interpretável ("risco dobrado em relação à média") e captura mais casos reais no grupo prioritário.
- **`percentil`:** `Alto` = top 10%, `Médio` = 10–30%. Lista mais curta, para quando a capacidade de atendimento é limitada.

In [ ]:
from sklearn.model_selection import cross_val_predict
# Probabilidades out-of-fold para toda a base
proba_oof = cross_val_predict(
    Pipeline([('preproc', preproc), ('clf', modelos[melhor_nome])]),
    X, y, groups=groups, cv=GroupKFold(5), method='predict_proba')[:, 1]
pares['proba_risco'] = proba_oof
br = y.mean()

if ESTRATEGIA_FAIXAS == 'base_rate':
    corte_baixo, corte_alto = br, 2*br
elif ESTRATEGIA_FAIXAS == 'percentil':
    corte_baixo, corte_alto = np.quantile(proba_oof, 0.70), np.quantile(proba_oof, 0.90)
print(f'Estratégia: {ESTRATEGIA_FAIXAS} | base rate={br:.3f}')
print(f'Cortes -> Baixo < {corte_baixo:.3f} <= Médio < {corte_alto:.3f} <= Alto')

pares['faixa_risco'] = pd.cut(pares['proba_risco'], [-1, corte_baixo, corte_alto, 2],
                              labels=['Baixo','Médio','Alto'])
resumo = pares.groupby('faixa_risco', observed=True).agg(
    alunos=('alvo','size'),
    taxa_agravamento_real=('alvo','mean'),
    casos_reais=('alvo','sum')).reindex(['Baixo','Médio','Alto'])
resumo['pct_casos_capturados'] = (resumo['casos_reais']/y.sum()*100).round(1)
print(); print(resumo.round(3))
print(f"\nGrupo Alto: {resumo.loc['Alto','alunos']} alunos, "
      f"{resumo.loc['Alto','taxa_agravamento_real']:.0%} agravam, "
      f"capta {resumo.loc['Alto','pct_casos_capturados']:.0f}% de todos os casos reais.")
print(f"Grupo Baixo: {resumo.loc['Baixo','alunos']} alunos, apenas "
      f"{resumo.loc['Baixo','taxa_agravamento_real']:.0%} agravam (pode receber menos atenção).")

## 11. Robustez temporal (out-of-time) e sensibilidade do IPP

O regime principal (por RA) responde se o modelo aprendeu o padrão; o corte temporal responde se ele funciona num ano que ainda não existia. Treino apenas com 2022→2023 e testo em 2023→2024, lendo o resultado com o limiar principal e com um limiar derivado só do treino — a diferença separa drift real de descalibração. A mesma célula registra a sensibilidade do IPP (mantido como feature) nos dois regimes.

In [ ]:
# Regime temporal: treina só com 2022->2023 e testa em 2023->2024 (ano nunca visto).
# Dois limiares na leitura: o principal (0,30, calibrado no regime por RA) e um derivado
# apenas do treino temporal via CV interna por RA — separa drift real de descalibração.
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score

m_tr = pares['Ano'].eq(2022)
m_te = pares['Ano'].eq(2023)

pipe_temp = Pipeline([('preproc', clone(preproc)), ('clf', clone(modelos[melhor_nome]))])
proba_tr = cross_val_predict(pipe_temp, X[m_tr], y[m_tr], groups=groups[m_tr],
                             cv=GroupKFold(5), method='predict_proba')[:, 1]
grade = np.arange(0.10, 0.71, 0.01)
limiar_temporal = round(float(grade[np.argmax([balanced_accuracy_score(y[m_tr], proba_tr >= t)
                                               for t in grade])]), 2)

pipe_temp.fit(X[m_tr], y[m_tr])
proba_te = pipe_temp.predict_proba(X[m_te])[:, 1]
auc_temporal = roc_auc_score(y[m_te], proba_te)
baseline_temporal = 1 - y[m_te].mean()

print(f'Treino 2022->2023: {m_tr.sum()} pares | Teste 2023->2024: {m_te.sum()} pares')
print(f'ROC-AUC out-of-time: {auc_temporal:.3f}   (regime principal, CV por RA: {scores.mean():.3f} +/- {scores.std():.3f})')
print(f'Baseline trivial do ano de teste: {baseline_temporal:.3f}\n')

met_temporal = {}
for rotulo, t in [('limiar_principal_0.30', LIMIAR_DECISAO),
                  (f'limiar_treino_temporal_{limiar_temporal:.2f}', limiar_temporal)]:
    pred_te = (proba_te >= t).astype(int)
    met_temporal[rotulo] = {'acuracia': round(accuracy_score(y[m_te], pred_te), 3),
                            'balanced_accuracy': round(balanced_accuracy_score(y[m_te], pred_te), 3)}
    print(f"{rotulo.replace('_',' ')}: acurácia {met_temporal[rotulo]['acuracia']:.3f} | "
          f"balanced accuracy {met_temporal[rotulo]['balanced_accuracy']:.3f}")

# Sensibilidade do IPP nos dois regimes. Decisão fechada: IPP permanece como feature
# (reconstrução algébrica exata sobre a fórmula documental do INDE); os números ficam
# registrados para o relatório.
num_sem_ipp = [c for c in num_features if c != 'IPP']
pre_sem = ColumnTransformer([('num', StandardScaler(), num_sem_ipp),
                             ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)])
pipe_sem = Pipeline([('preproc', pre_sem), ('clf', clone(modelos[melhor_nome]))])
auc_ra_sem = cross_val_score(pipe_sem, pares[num_sem_ipp + cat_features], y, groups=groups,
                             cv=GroupKFold(5), scoring='roc_auc').mean()
pipe_sem.fit(pares.loc[m_tr, num_sem_ipp + cat_features], y[m_tr])
auc_temp_sem = roc_auc_score(y[m_te], pipe_sem.predict_proba(pares.loc[m_te, num_sem_ipp + cat_features])[:, 1])
print(f'\nSensibilidade IPP — AUC CV por RA: com {scores.mean():.3f} / sem {auc_ra_sem:.3f} | '
      f'AUC temporal: com {auc_temporal:.3f} / sem {auc_temp_sem:.3f}')

## 12. Persistência do modelo (para o app Streamlit)

In [ ]:
import joblib, json
pipe_final = Pipeline([('preproc', preproc), ('clf', modelos[melhor_nome])]).fit(X, y)
caminho_pkl = os.path.join(PASTA_DRIVE, 'modelo_risco_defasagem.pkl')
caminho_meta = os.path.join(PASTA_DRIVE, 'modelo_meta.json')
joblib.dump(pipe_final, caminho_pkl)
meta = {'alvo':'agravamento_defasagem_T_para_T1',
        'num_features':num_features, 'cat_features':cat_features,
        'modelo':melhor_nome, 'split':ESTRATEGIA_SPLIT,
        'robustez_temporal': {'auc_out_of_time': round(float(auc_temporal),3), 'baseline_trivial': round(float(baseline_temporal),3), 'limiar_recalibrado_treino': limiar_temporal, 'metricas_por_limiar': met_temporal},
        'sensibilidade_ipp': {'auc_cv_por_RA_sem_ipp': round(float(auc_ra_sem),3), 'auc_temporal_sem_ipp': round(float(auc_temp_sem),3)},
        'auc_cv': round(float(scores.mean()),3),
        'estrategia_faixas': ESTRATEGIA_FAIXAS,
        'corte_medio': round(float(corte_baixo),3),
        'corte_alto': round(float(corte_alto),3)}
with open(caminho_meta,'w') as f: json.dump(meta,f,indent=2,ensure_ascii=False)
print('Salvos em:', PASTA_DRIVE)
print(' -', caminho_pkl); print(' -', caminho_meta)

---
## Decisões técnicas (para o storytelling / defesa na banca)

1. **Alvo = agravamento de defasagem (T→T+1).** Fiel ao texto do enunciado, que nomeia "risco de defasagem" duas vezes. Modelo temporal/prospectivo, features do ano anterior → sem vazamento.
2. **O modelo É a resposta a "quais padrões".** SHAP + permutation importance sobre o modelo treinado identificam os indicadores de risco. Não há análise descritiva paralela — o instrumento preditivo e o analítico são o mesmo.
3. **Interpretação em duas camadas (estrutural vs. acionável).** O AUC alto tem componente estrutural (defasagem/fase são previsíveis por construção). O valor real está no **incremento dos índices comportamentais/psicossociais** — quantificado na seção 8. Reportar isso é honestidade metodológica que a banca cobra.
4. **Features enxutas, sem redundância.** IAN (vazamento), INDE_ano (redundante/colinear) e idade (colinear com fase) fora do modelo. Índices puros preservam a interpretabilidade do SHAP.
5. **Sem imputação; split sem vazamento de grupo; acurácia descartada** (base desbalanceada) em favor de ROC-AUC / PR-AUC.
